# Gradient Boosting for Classification (Geometric Intuition)

Gradient Boosting for Classification follows the same **additive modeling** principle as Gradient Boosting for Regression. Instead of fitting each new model to the original target labels, every new model learns the **pseudo-residuals (errors)** produced by the current ensemble.

The major difference lies in the **loss function**. Regression uses **Mean Squared Error (MSE)**, whereas Classification uses **Log Loss (Binary Cross-Entropy)**. Consequently, Gradient Boosting for Classification operates in **Log-Odds space**, converting predictions back to probabilities using the **Sigmoid function** whenever probabilities are required.

---

# 1. Additive Modeling

Gradient Boosting builds the final classifier by sequentially adding multiple weak learners.

$$
F(x)=F_0(x)+F_1(x)+F_2(x)+\cdots+F_M(x)
$$

where

* $F_0(x)$ = Initial model
* $F_1(x),F_2(x),\ldots$ = Sequential regression trees
* $M$ = Number of boosting iterations

Each new tree attempts to reduce the errors made by the previous ensemble.

---

# 2. Regression vs Classification

| Aspect             | Regression               | Classification                  |
| ------------------ | ------------------------ | ------------------------------- |
| Target             | Continuous values        | Binary labels (0/1)             |
| Loss Function      | Mean Squared Error (MSE) | Binary Cross-Entropy (Log Loss) |
| Initial Prediction | Mean of target           | Log-Odds                        |
| Prediction Space   | Continuous output        | Log-Odds                        |
| Final Output       | Numeric value            | Probability using Sigmoid       |

---

# 3. Stage 1 — Initial Base Model

Unlike regression, the first model is **not the mean of the target values**.

Instead, Gradient Boosting Classification computes the **Log-Odds** of the dataset.

## Initial Log-Odds

$$
\boxed{
\text{Log-Odds}
=
\log\left(
\frac{\text{Number of Positive Samples}}
{\text{Number of Negative Samples}}
\right)
}
$$
-
### Example

Suppose

* Positive Class = 5
* Negative Class = 3

Then

$$
\text{Log-Odds}
=
\log\left(\frac{5}{3}\right)
\approx0.51
$$
-
## Convert Log-Odds into Probability

The Log-Odds are converted into probabilities using the **Sigmoid Function**.

$$
\boxed{
P=
\frac{e^{\text{Log-Odds}}}
{1+e^{\text{Log-Odds}}}
}
$$

Substituting

$$
\text{Log-Odds}=0.51
$$

gives

$$
P=0.625
$$

Therefore, every training sample initially receives the same prediction

$$
\boxed{
P=0.625
}
$$

---

# 4. Stage 2 — Train the First Regression Tree

Although the task is classification, Gradient Boosting trains a **Regression Tree** because the residuals are continuous values.

## Step 1 — Compute Pseudo-Residuals

The residual for every training sample is

$$
\boxed{
\text{Residual}
=
Y-P
}
$$
-
where

* $Y$ = Actual class
* $P$ = Current predicted probability

### Example

For a positive sample

$$
1-0.625=0.375
$$

For a negative sample

$$
0-0.625=-0.625
$$

## Step 2 — Train a Regression Tree

Train a regression tree using

* Input Features
* Residuals

The regression tree learns patterns in the residuals instead of the original labels.

## Step 3 — Compute Leaf Node Output

The regression tree predicts residuals.

However, the ensemble prediction is maintained in **Log-Odds**, so each leaf prediction must be converted.

The optimal leaf value is

$$
\boxed{
\gamma
=
\frac{\sum \text{Residuals}}
{\sum P_{\text{previous}}\left(1-P_{\text{previous}}\right)}
}
$$
-
where

* Numerator = Sum of residuals inside the leaf
* Denominator = Sum of $P(1-P)$ for every sample inside the leaf

## Step 4 — Update the Log-Odds

The prediction becomes

$$
\boxed{
F_1(x)
=
F_0(x)+\alpha\gamma
}
$$
-
where

* $\alpha$ = Learning Rate
* $\gamma$ = Leaf Output

The learning rate controls how much the new tree influences the ensemble.

Typical value:

$$
\alpha=0.1
$$

---

# 5. Stage 3 — Train the Second Regression Tree

The updated Log-Odds are converted back into probabilities.

## Updated Probability

$$
P=
\frac{e^{F_1(x)}}
{1+e^{F_1(x)}}
$$

## Compute New Residuals

$$
\boxed{
\text{Residual}_{\text{new}}
=
Y-P_{\text{updated}}
}
$$
-
Notice that these residuals are now closer to zero because the model has improved.

## Train Another Regression Tree

A second regression tree is fitted using these updated residuals.

The ensemble becomes

$$
\boxed{
F_2(x)
=
F_0(x)
+\alpha\gamma_1
+\alpha\gamma_2
}
$$
-
This process continues until the desired number of trees has been trained.

---

# 6. Prediction on New Data

Suppose a new student has

* CGPA = 7.2
* IQ = 100

Prediction proceeds as follows:

### Step 1

Start with the baseline Log-Odds

$$
F_0(x)
$$

### Step 2

Traverse Tree 1

Add

$$
\alpha\gamma_1
$$

### Step 3

Traverse Tree 2

Add

$$
\alpha\gamma_2
$$

### Step 4

Continue through every tree

$$
F(x)
=
F_0(x)
+\alpha\sum_{m=1}^{M}\gamma_m
$$
-
### Step 5

Convert Log-Odds into Probability

$$
\boxed{
P=
\frac{e^{F(x)}}
{1+e^{F(x)}}
}
$$

### Step 6

Final Classification

$$
\boxed{
\begin{cases}
P\ge0.5,&\text{Class 1}\
P<0.5,&\text{Class 0}
\end{cases}
}
$$

---

# 7. Geometric Intuition

Gradient Boosting can be visualized as gradually **warping a flat prediction surface**.

### Initial Surface

The first model creates a flat plane where every point has the same probability.

### First Tree

The first regression tree bends the plane upward in regions where the model underestimates and downward where it overestimates.

### Subsequent Trees

Each new tree further adjusts the prediction surface.

Instead of making large corrections, every tree introduces a small adjustment controlled by the learning rate.

### Final Surface

After many boosting iterations, the initially flat surface becomes highly flexible, accurately separating complex decision boundaries.

---

# 8. Complete Algorithm

1. Compute the initial Log-Odds.
2. Convert Log-Odds into probabilities.
3. Compute pseudo-residuals.
4. Train a regression tree on residuals.
5. Compute the optimal leaf outputs.
6. Update Log-Odds.
7. Convert updated Log-Odds into probabilities.
8. Repeat until all trees have been trained.
9. Convert the final Log-Odds into probabilities for prediction.

---

# 9. Summary Table

| Hyperparameter / Concept | Summary                                                           |
| ------------------------ | ----------------------------------------------------------------- |
| Additive Modeling        | Builds the model by sequentially adding weak learners.            |
| Initial Model            | Uses Log-Odds instead of the target mean.                         |
| Log-Odds                 | $\log\left(\frac{\text{Positive}}{\text{Negative}}\right)$        |
| Sigmoid Function         | Converts Log-Odds into probabilities.                             |
| Pseudo-Residual          | $Y-P$                                                             |
| Weak Learner             | Regression Tree                                                   |
| Leaf Output ($\gamma$)   | Converts residual predictions into Log-Odds updates.              |
| Learning Rate ($\alpha$) | Controls how much each tree contributes.                          |
| Ensemble Update          | $F_m(x)=F_{m-1}(x)+\alpha\gamma$                                  |
| Final Prediction         | Convert final Log-Odds to probability using the Sigmoid function. |
| Final Classification     | Probability $\ge 0.5$ → Class 1, otherwise Class 0.               |


### Blog  
https://towardsdatascience.com/all-you-need-to-know-about-gradient-boosting-algorithm-part-2-classification-d3ed8f56541e/ 

### Code 

https://colab.research.google.com/drive/13p46IFhg3h6BIdjxUcfXPco13jIOCV6I?usp=sharing#scrollTo=9fd07a7d